In [14]:
from backtesting.test import GOOG

In [15]:
GOOG.head()

,Open,High,Low,Close,Volume
2004-08-19,100.00,104.06,95.96,100.34,22351900
2004-08-20,101.01,109.08,100.50,108.31,11428600
2004-08-23,110.75,113.48,109.05,109.40,9137200
2004-08-24,111.24,111.60,103.57,104.87,7631300
2004-08-25,104.96,108.00,103.88,106.00,4598900


In [16]:
GOOG.tail()

,Open,High,Low,Close,Volume
2013-02-25,802.3,808.41,790.49,790.77,2303900
2013-02-26,795.0,795.95,784.40,790.13,2202500
2013-02-27,794.8,804.75,791.11,799.78,2026100
2013-02-28,801.1,806.99,801.03,801.20,2265800
2013-03-01,797.8,807.14,796.15,806.19,2175400


In [17]:
def SMA(values, n):
    """
    Return simple moving average of `values`, at
    each step taking into account `n` previous values.
    """
    result = []
    for i, v in enumerate(values):
        if i < n - 1:
            result.append(float('nan'))
            continue
        sma = sum(values[i-n+1:i+1]) / n
        result.append(sma)
    return result

# def SMA(values, n):
#     """
#     Return simple moving average of `values`, at
#     each step taking into account `n` previous values.
#     """
#     return pd.Series(values).rolling(n).mean()

In [18]:
from backtesting import Strategy
from backtesting.lib import crossover

class SmaCross(Strategy):
    # Define the two MA lags as *class variables*
    # for later optimization
    n1 = 10
    n2 = 20
    
    def init(self):
        # Precompute the two moving averages
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)
    
    def next(self):
        # Note to self: The last array value (e.g. data.Close[-1]) is always the most recent value.
        
        # If sma1 crosses above sma2, close any existing
        # short trades, and buy the asset
        if crossover(self.sma1, self.sma2):
            self.position.close()
            self.buy()

        # Else, if sma1 crosses below sma2, close any existing
        # long trades, and sell the asset
        elif crossover(self.sma2, self.sma1):
            self.position.close()
            self.sell()

In [19]:
from backtesting import Backtest

bt = Backtest(GOOG, SmaCross, cash=10_000, commission=.002, finalize_trades=True)
stats = bt.run()
stats

Start                     2004-08-19 00:00:00
End                       2013-03-01 00:00:00
Duration                   3116 days 00:00:00
Exposure Time [%]                    97.06704
Equity Final [$]                  55574.51294
Equity Peak [$]                   56309.05934
Commissions [$]                   10770.95706
Return [%]                          455.74513
Buy & Hold Return [%]               607.37036
Return (Ann.) [%]                    22.30053
Volatility (Ann.) [%]                37.36319
CAGR [%]                             22.26792
Sharpe Ratio                          0.59686
Sortino Ratio                           1.136
Calmar Ratio                          0.65722
Alpha [%]                           443.77072
Beta                                  0.01972
Max. Drawdown [%]                   -33.93159
Avg. Drawdown [%]                    -6.16072
Max. Drawdown Duration      830 days 00:00:00
Avg. Drawdown Duration       50 days 00:00:00
# Trades                          

In [25]:
a = bt.plot(superimpose=False)

/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_scroll' property; using the latest value
  fig = gridplot(


In [21]:
%%time

stats = bt.optimize(n1=range(5, 30, 5),
                    n2=range(10, 70, 5),
                    maximize='Equity Final [$]',
                    constraint=lambda param: param.n1 < param.n2)
stats

/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/backtesting.py:1639: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


CPU times: user 1.28 s, sys: 302 ms, total: 1.59 s
Wall time: 1.41 s


Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/psm_707878f7'
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/psm_d4e10784'
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/psm_3ea04b0e'
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyErr

Start                     2004-08-19 00:00:00
End                       2013-03-01 00:00:00
Duration                   3116 days 00:00:00
Exposure Time [%]                     99.0689
Equity Final [$]                  76870.43702
Equity Peak [$]                    84982.1921
Commissions [$]                   31068.31298
Return [%]                          668.70437
Buy & Hold Return [%]               687.98749
Return (Ann.) [%]                    27.04697
Volatility (Ann.) [%]                43.15387
CAGR [%]                             27.00668
Sharpe Ratio                          0.62676
Sortino Ratio                         1.27047
Calmar Ratio                          0.60706
Alpha [%]                           605.25971
Beta                                  0.09222
Max. Drawdown [%]                   -44.55387
Avg. Drawdown [%]                    -5.81107
Max. Drawdown Duration     1558 days 00:00:00
Avg. Drawdown Duration       50 days 00:00:00
# Trades                          

In [22]:
stats._strategy

<Strategy SmaCross(n1=10,n2=15)>

In [23]:
bt.plot(plot_volume=False, plot_pl=False, superimpose=False)

/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_scroll' property; using the latest value
  fig = gridplot(


GridPlot(id='p2126', ...)